In [ ]:
"""
Завдання 3. Вплив заповнення пропущених значень.

Порівнюємо базову модель на повних даних із моделлю,
для якої в одній ознаці штучно створено пропуски,
а пропуски заповнюються середнім значенням через SimpleImputer.
"""

In [2]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import r2_score

X, y = make_regression(
    n_samples=2000,
    n_features=4,
    noise=10,
    random_state=42
)
X_train_full, X_test_full, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

baseline_model = LinearRegression()
baseline_model.fit(X_train_full, y_train)
y_pred_baseline = baseline_model.predict(X_test_full)
r2_baseline = r2_score(y_test, y_pred_baseline)

In [3]:
X_with_missing = X.copy()
rng = np.random.default_rng(42)

missing_mask = rng.random(X_with_missing.shape[0]) < 0.20
X_with_missing[missing_mask, 0] = np.nan

# Поділ даних із пропусками на ті самі train/test індекси
# Використовуємо індекси, щоб цільові значення відповідали рядкам.
indices = np.arange(len(y))
train_indices, test_indices = train_test_split(
    indices,
    test_size=0.2,
    random_state=42
)

X_train_missing = X_with_missing[train_indices]
X_test_missing = X_with_missing[test_indices]
y_train_missing = y[train_indices]
y_test_missing = y[test_indices]


In [4]:
imputed_model = make_pipeline(
    SimpleImputer(strategy="mean"),
    LinearRegression()
)
imputed_model.fit(X_train_missing, y_train_missing)
y_pred_imputed = imputed_model.predict(X_test_missing)
r2_imputed = r2_score(y_test_missing, y_pred_imputed)

results = pd.DataFrame({
    "Сценарій": ["Без пропусків", "Після імпутації"],
    "R²": [r2_baseline, r2_imputed]
})

print(results.to_string(index=False, formatters={"R²": "{:.6f}".format}))
print(f"\nЧастка пропусків у першій ознаці: {missing_mask.mean():.1%}")

       Сценарій       R²
  Без пропусків 0.993724
Після імпутації 0.993639

Частка пропусків у першій ознаці: 20.4%
